In [19]:
import numpy as np
from LanzaModels import TVL2_1D
from ADMMsRustici import SpectralSolver
from signalClass import *
import time

In [20]:
np.random.seed(24102000)
n = 512

construct blur matrix

In [21]:
#blur matrix construction

a = 0.25
b = 0.5
c = 0.25

diagB = b * np.ones(shape=(n,))
offDiagA = a * np.ones(shape=(n-1,))
offDiagC = c * np.ones(shape=(n-1,))
A = np.diag(diagB, 0) + np.diag(offDiagC, 1) + np.diag(offDiagA, -1)

#apply anti-reflexive BCs

A[0][0] = 2 * a + b
A[0][1] = c - a
A[n-1][n-2] = a - c
A[n-1][n-1] = b + 2 * c

#end blur matrix construction

construct signal

In [22]:
#begin signal construction

PwSignal = signal(n)
RndSignal = signal(n)
sigma = 0.01

PwSignal.generate_cartoon_sign(2, 50)
RndSignal.generate_GG_realization(0, sigma, 1)

xTrue = PwSignal.get_image()
xCorrupted = (A @ xTrue) + RndSignal.get_image()

#end signal construction

Define the TVL2 model

In [23]:
mu = 2
VarModel = TVL2_1D.TVL2_1DClass(A, xCorrupted, mu)

Now, we need to initialize and define the solver

In [24]:
#begin solver construction
np.random.seed(24102001)

xk = np.random.randn(n,)
yk = np.random.randn(n,)
betak = 1
lk = np.zeros(n)

x0 = xk.copy()
y0 = yk.copy()

MySolver = SpectralSolver.SpectralSolverClass(VarModel, xk, yk, lk, betak)

#end solver construction

In [25]:
iters = 1900

XsolutionHistory = np.zeros(shape=(iters, n))
YsolutionHistory = np.zeros(shape=(iters, n))

lambdaHistory = np.zeros(shape=(iters, n))

betaHistory = np.zeros(shape=(iters,))

PrimalResidueHistory = np.zeros(shape=(iters,))
DualResidueHistory = np.zeros(shape=(iters,))
ImgHistory = np.zeros(shape=(iters,))

CpuTimes = np.zeros(shape=(iters,))

In [26]:
timer = 0

for iter in range(0, iters):

    print(f"{iter + 1} / {iters}")

    sTime = time.perf_counter_ns()

    xk_1, yk_1, lk_1, betak_1 = MySolver.CallIterationStep(xk, yk, lk, betak)

    eTime = time.perf_counter_ns()

    timer += ( (eTime - sTime) / 1e9 )

    
    primalResidue = np.linalg.norm(VarModel.P @ xk_1 + VarModel.Q @ yk_1 - VarModel.c)
    dualResidue = betak_1 * np.linalg.norm(VarModel.P.T @ (VarModel.Q @ (yk - yk_1)) )

    XsolutionHistory[iter, :] = xk_1
    YsolutionHistory[iter, :] = yk_1
    lambdaHistory[iter, :] = lk_1
    betaHistory[iter] = betak_1

    PrimalResidueHistory[iter] = primalResidue
    DualResidueHistory[iter] = dualResidue
    ImgHistory[iter] = VarModel(xk_1)
    CpuTimes[iter] = timer

    xk = xk_1
    yk = yk_1
    lk = lk_1
    betak = betak_1

    if (max(primalResidue, dualResidue) <= 1e-9):
        print(iter)
        break


1 / 1900
2 / 1900
3 / 1900
4 / 1900
5 / 1900
6 / 1900
7 / 1900
8 / 1900
9 / 1900
10 / 1900
11 / 1900
12 / 1900
13 / 1900
14 / 1900
15 / 1900
16 / 1900
17 / 1900
18 / 1900
19 / 1900
20 / 1900
21 / 1900
22 / 1900
23 / 1900
24 / 1900
25 / 1900
26 / 1900
27 / 1900
28 / 1900
29 / 1900
30 / 1900
31 / 1900
32 / 1900
33 / 1900
34 / 1900
35 / 1900
36 / 1900
37 / 1900
38 / 1900
39 / 1900
40 / 1900
41 / 1900
42 / 1900
43 / 1900
44 / 1900
45 / 1900
46 / 1900
47 / 1900
48 / 1900
49 / 1900
50 / 1900
51 / 1900
52 / 1900
53 / 1900
54 / 1900
55 / 1900
56 / 1900
57 / 1900
58 / 1900
59 / 1900
60 / 1900
61 / 1900
62 / 1900
63 / 1900
64 / 1900
65 / 1900
66 / 1900
67 / 1900
68 / 1900
69 / 1900
70 / 1900
71 / 1900
72 / 1900
73 / 1900
74 / 1900
75 / 1900
76 / 1900
77 / 1900
78 / 1900
79 / 1900
80 / 1900
81 / 1900
82 / 1900
83 / 1900
84 / 1900
85 / 1900
86 / 1900
87 / 1900
88 / 1900
89 / 1900
90 / 1900
91 / 1900
92 / 1900
93 / 1900
94 / 1900
95 / 1900
96 / 1900
97 / 1900
98 / 1900
99 / 1900
100 / 1900
101 / 19

In [27]:
np.savez_compressed(
    "./SpectralADMMTVL2-Laplace.npz",
    Xs = XsolutionHistory,
    Ys = YsolutionHistory,
    Ls = lambdaHistory,
    Betas = betaHistory,

    PrimalRes = PrimalResidueHistory,
    DualRes = DualResidueHistory,
	IMGs = ImgHistory,
    CpuTimes = CpuTimes,

    xTrue = xTrue,
    xCorrupted = xCorrupted,
	x0 = x0,
	y0 = y0
)